# ZGLMM159 — M2 — Análise Exploratória

**Tabela:** `prd_procurement.corp_curated.vw_zglmm159_m2`
**Ambiente:** 🔴 **PRODUÇÃO (PRD)** · **Objeto:** VIEW
**Domínio:** Dados MRP 2 por material e centro (suprimento, estoque de seguranca, prazos)
**Colunas:** 28 · **Parte M2** de 3 (C1 · M1 · M2)

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** e **somente leitura** — nenhum `CREATE`,
`INSERT` ou gravação. Sem widget, sem view temporária, sem ordem obrigatória.

## Alertas de partida _(dos comentários do `DESCRIBE` e da validação anterior)_

| Severidade | Alerta |
|---|---|
| 🔴 ALTA | **10 colunas declaradas sem fonte** (*"Sem fonte nas cinco cleansed; NULL"*): `ind_remessa_jit`, `ind_entrada_lotes`, `cod_calendario_planejamento`, `cod_perfil_cobertura`, `cod_margem_seguranca`, `cod_perfil_periodo`, `ind_utilizacao_quotizacao`, `ind_aap`, `qt_estoque_minimo_seguranca`, `pct_grau_atendimento`. Estarão 100% nulas — no SAP podem ter valor. **Limitação de escopo** a registrar, não erro de carga (seção 6.1). |
| 🟡 MÉDIA | **`ind_marcado_exclusao` preservado sem filtro** — a view traz também itens marcados para eliminação. Se a ZGLMM159 no SAP filtra eliminados, a comparação acusará sobra (seção 16). |
| 🟡 MÉDIA | **Nenhuma coluna de auditoria de carga** (sem `dateingest`/`dh_carga`) — não é possível saber a data do snapshot. Anotar data/hora da consulta. |
| ⚪ BAIXA | **Objeto é VIEW** — `DESCRIBE DETAIL`/`HISTORY` comentados para não quebrar o Run All. |

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados |
| 2 | Volumetria |
| 3 | Distribuição por centro — escolha do cenário |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| **6** | **Preenchimento de todas as colunas** |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 10 | Perfil numérico |
| **11** | **Totais para conciliação** |
| 12 | Datas |
| 13 | Códigos e zeros à esquerda |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| **18** | **Análises específicas + cruzamento C1 × M1 × M2** |
| 19 | Amostra ampliada |
| 20 | Resumo |


## 1. Metadados

In [0]:
-- 1. ESTRUTURA
DESCRIBE EXTENDED prd_procurement.corp_curated.vw_zglmm159_m2;

In [0]:
-- 1.1 Detalhe e historico nao se aplicam a view.
-- DESCRIBE DETAIL prd_procurement.corp_curated.vw_zglmm159_m2;
-- DESCRIBE HISTORY prd_procurement.corp_curated.vw_zglmm159_m2 LIMIT 20;
SELECT 'Objeto e uma VIEW — os dois comandos acima nao se aplicam' AS observacao;

## 2. Volumetria

In [0]:
-- 2. VOLUMETRIA
SELECT COUNT(*) AS linhas,
       COUNT(DISTINCT cod_material) AS materiais,
       COUNT(DISTINCT cod_centro) AS centros,
       COUNT(DISTINCT cod_empresa) AS empresas,
       COUNT(DISTINCT CONCAT_WS('|', cod_material, cod_centro)) AS chaves_material_centro
FROM prd_procurement.corp_curated.vw_zglmm159_m2;

## 3. Distribuição por centro

Base para o cenário de teste (10 mil a 300 mil linhas). A 3.1 mostra os centros já usados
nos demais testes (4128, 4014) e o 4058 da validação anterior.

In [0]:
-- 3. DISTRIBUICAO POR CENTRO
SELECT COALESCE(NULLIF(trim(cod_centro),''),'(vazio)') AS cod_centro, COUNT(*) AS linhas,
       COUNT(DISTINCT cod_material) AS materiais,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_m2), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000 THEN 'CANDIDATO A CENARIO' ELSE '' END AS sugestao
FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY 1 ORDER BY linhas DESC LIMIT 60;

In [0]:
-- 3.1 CENTROS DOS DEMAIS TESTES
SELECT cod_centro, COUNT(*) AS linhas, COUNT(DISTINCT cod_material) AS materiais,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_m2), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro IN ('4128', '4014', '4058')
GROUP BY cod_centro ORDER BY cod_centro;

## 4. Granularidade e chave real

In [0]:
-- 4. GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_zglmm159_m2),
g AS (
  SELECT 'cod_material' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material` FROM prd_procurement.corp_curated.vw_zglmm159_m2)
UNION ALL
  SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_centro` FROM prd_procurement.corp_curated.vw_zglmm159_m2)
UNION ALL
  SELECT 'cod_material + cod_centro + cod_empresa' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_empresa` FROM prd_procurement.corp_curated.vw_zglmm159_m2)
)
SELECT g.chave, t.total AS linhas, g.distintos, ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA' ELSE 'NAO UNICA' END AS veredito
FROM g CROSS JOIN t ORDER BY linhas_por_chave;

## 5. Duplicidade

Pela chave esperada `cod_material + cod_centro`.

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `cod_material`, `cod_centro`, COUNT(*) AS qtd FROM prd_procurement.corp_curated.vw_zglmm159_m2
GROUP BY `cod_material`, `cod_centro` HAVING COUNT(*) > 1 ORDER BY qtd DESC LIMIT 30;

In [0]:
-- 5.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH dup AS (SELECT `cod_material`, `cod_centro` FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY `cod_material`, `cod_centro` HAVING COUNT(*) > 1),
d AS (SELECT c.* FROM prd_procurement.corp_curated.vw_zglmm159_m2 c JOIN dup ON c.`cod_material` <=> dup.`cod_material` AND c.`cod_centro` <=> dup.`cod_centro`),
agg AS (SELECT `cod_material`, `cod_centro`,
         COUNT(DISTINCT `cod_empresa`) AS `cod_empresa`,
         COUNT(DISTINCT `tp_suprimento`) AS `tp_suprimento`,
         COUNT(DISTINCT `tp_grupo_determinacao_estoque`) AS `tp_grupo_determinacao_estoque`,
         COUNT(DISTINCT `ind_remessa_jit`) AS `ind_remessa_jit`,
         COUNT(DISTINCT `cod_chave_horizonte`) AS `cod_chave_horizonte`,
         COUNT(DISTINCT `ind_entrada_lotes`) AS `ind_entrada_lotes`,
         COUNT(DISTINCT `ind_coproduto`) AS `ind_coproduto`,
         COUNT(DISTINCT `cod_deposito_suprimento_externo`) AS `cod_deposito_suprimento_externo`,
         COUNT(DISTINCT `cod_deposito_producao`) AS `cod_deposito_producao`,
         COUNT(DISTINCT `cod_calendario_planejamento`) AS `cod_calendario_planejamento`,
         COUNT(DISTINCT `ind_baixa_explosao`) AS `ind_baixa_explosao`,
         COUNT(DISTINCT `cod_perfil_cobertura`) AS `cod_perfil_cobertura`,
         COUNT(DISTINCT `ind_material_granel`) AS `ind_material_granel`,
         COUNT(DISTINCT `cod_margem_seguranca`) AS `cod_margem_seguranca`,
         COUNT(DISTINCT `cod_perfil_periodo`) AS `cod_perfil_periodo`,
         COUNT(DISTINCT `qt_margem_seguranca`) AS `qt_margem_seguranca`,
         COUNT(DISTINCT `tp_suprimento_especial`) AS `tp_suprimento_especial`,
         COUNT(DISTINCT `ind_utilizacao_quotizacao`) AS `ind_utilizacao_quotizacao`,
         COUNT(DISTINCT `ind_aap`) AS `ind_aap`,
         COUNT(DISTINCT `qt_tempo_producao_interna`) AS `qt_tempo_producao_interna`,
         COUNT(DISTINCT `qt_estoque_seguranca`) AS `qt_estoque_seguranca`,
         COUNT(DISTINCT `qt_estoque_minimo_seguranca`) AS `qt_estoque_minimo_seguranca`,
         COUNT(DISTINCT `pct_grau_atendimento`) AS `pct_grau_atendimento`,
         COUNT(DISTINCT `qt_dias_prazo_entrega_previsto`) AS `qt_dias_prazo_entrega_previsto`,
         COUNT(DISTINCT `qt_dias_processamento_entrada`) AS `qt_dias_processamento_entrada`,
         COUNT(DISTINCT `ind_marcado_exclusao`) AS `ind_marcado_exclusao`
  FROM d GROUP BY `cod_material`, `cod_centro`)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real' ELSE 'constante' END AS veredito
FROM (SELECT stack(26,
    'cod_empresa', MAX(`cod_empresa`),
    'tp_suprimento', MAX(`tp_suprimento`),
    'tp_grupo_determinacao_estoque', MAX(`tp_grupo_determinacao_estoque`),
    'ind_remessa_jit', MAX(`ind_remessa_jit`),
    'cod_chave_horizonte', MAX(`cod_chave_horizonte`),
    'ind_entrada_lotes', MAX(`ind_entrada_lotes`),
    'ind_coproduto', MAX(`ind_coproduto`),
    'cod_deposito_suprimento_externo', MAX(`cod_deposito_suprimento_externo`),
    'cod_deposito_producao', MAX(`cod_deposito_producao`),
    'cod_calendario_planejamento', MAX(`cod_calendario_planejamento`),
    'ind_baixa_explosao', MAX(`ind_baixa_explosao`),
    'cod_perfil_cobertura', MAX(`cod_perfil_cobertura`),
    'ind_material_granel', MAX(`ind_material_granel`),
    'cod_margem_seguranca', MAX(`cod_margem_seguranca`),
    'cod_perfil_periodo', MAX(`cod_perfil_periodo`),
    'qt_margem_seguranca', MAX(`qt_margem_seguranca`),
    'tp_suprimento_especial', MAX(`tp_suprimento_especial`),
    'ind_utilizacao_quotizacao', MAX(`ind_utilizacao_quotizacao`),
    'ind_aap', MAX(`ind_aap`),
    'qt_tempo_producao_interna', MAX(`qt_tempo_producao_interna`),
    'qt_estoque_seguranca', MAX(`qt_estoque_seguranca`),
    'qt_estoque_minimo_seguranca', MAX(`qt_estoque_minimo_seguranca`),
    'pct_grau_atendimento', MAX(`pct_grau_atendimento`),
    'qt_dias_prazo_entrega_previsto', MAX(`qt_dias_prazo_entrega_previsto`),
    'qt_dias_processamento_entrada', MAX(`qt_dias_processamento_entrada`),
    'ind_marcado_exclusao', MAX(`ind_marcado_exclusao`)
  ) AS (coluna, max_valores_distintos) FROM agg)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Coluna 100% nula que **não** está na lista de "sem fonte" é erro de carga.

In [0]:
-- 6. PREENCHIMENTO
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_zglmm159_m2),
perf AS (SELECT stack(28,
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_empresa', 'string', COUNT_IF(`cod_empresa` IS NULL), COUNT_IF(`cod_empresa` IS NOT NULL AND lower(trim(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa`) RLIKE '^0+([.,]0+)?$'),
    'tp_suprimento', 'string', COUNT_IF(`tp_suprimento` IS NULL), COUNT_IF(`tp_suprimento` IS NOT NULL AND lower(trim(`tp_suprimento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_suprimento`) RLIKE '^0+([.,]0+)?$'),
    'tp_grupo_determinacao_estoque', 'string', COUNT_IF(`tp_grupo_determinacao_estoque` IS NULL), COUNT_IF(`tp_grupo_determinacao_estoque` IS NOT NULL AND lower(trim(`tp_grupo_determinacao_estoque`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_grupo_determinacao_estoque`) RLIKE '^0+([.,]0+)?$'),
    'ind_remessa_jit', 'string', COUNT_IF(`ind_remessa_jit` IS NULL), COUNT_IF(`ind_remessa_jit` IS NOT NULL AND lower(trim(`ind_remessa_jit`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_remessa_jit`) RLIKE '^0+([.,]0+)?$'),
    'cod_chave_horizonte', 'string', COUNT_IF(`cod_chave_horizonte` IS NULL), COUNT_IF(`cod_chave_horizonte` IS NOT NULL AND lower(trim(`cod_chave_horizonte`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_chave_horizonte`) RLIKE '^0+([.,]0+)?$'),
    'ind_entrada_lotes', 'string', COUNT_IF(`ind_entrada_lotes` IS NULL), COUNT_IF(`ind_entrada_lotes` IS NOT NULL AND lower(trim(`ind_entrada_lotes`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_entrada_lotes`) RLIKE '^0+([.,]0+)?$'),
    'ind_coproduto', 'string', COUNT_IF(`ind_coproduto` IS NULL), COUNT_IF(`ind_coproduto` IS NOT NULL AND lower(trim(`ind_coproduto`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_coproduto`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito_suprimento_externo', 'string', COUNT_IF(`cod_deposito_suprimento_externo` IS NULL), COUNT_IF(`cod_deposito_suprimento_externo` IS NOT NULL AND lower(trim(`cod_deposito_suprimento_externo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito_suprimento_externo`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito_producao', 'string', COUNT_IF(`cod_deposito_producao` IS NULL), COUNT_IF(`cod_deposito_producao` IS NOT NULL AND lower(trim(`cod_deposito_producao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito_producao`) RLIKE '^0+([.,]0+)?$'),
    'cod_calendario_planejamento', 'string', COUNT_IF(`cod_calendario_planejamento` IS NULL), COUNT_IF(`cod_calendario_planejamento` IS NOT NULL AND lower(trim(`cod_calendario_planejamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_calendario_planejamento`) RLIKE '^0+([.,]0+)?$'),
    'ind_baixa_explosao', 'string', COUNT_IF(`ind_baixa_explosao` IS NULL), COUNT_IF(`ind_baixa_explosao` IS NOT NULL AND lower(trim(`ind_baixa_explosao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_baixa_explosao`) RLIKE '^0+([.,]0+)?$'),
    'cod_perfil_cobertura', 'string', COUNT_IF(`cod_perfil_cobertura` IS NULL), COUNT_IF(`cod_perfil_cobertura` IS NOT NULL AND lower(trim(`cod_perfil_cobertura`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_perfil_cobertura`) RLIKE '^0+([.,]0+)?$'),
    'ind_material_granel', 'string', COUNT_IF(`ind_material_granel` IS NULL), COUNT_IF(`ind_material_granel` IS NOT NULL AND lower(trim(`ind_material_granel`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_material_granel`) RLIKE '^0+([.,]0+)?$'),
    'cod_margem_seguranca', 'string', COUNT_IF(`cod_margem_seguranca` IS NULL), COUNT_IF(`cod_margem_seguranca` IS NOT NULL AND lower(trim(`cod_margem_seguranca`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_margem_seguranca`) RLIKE '^0+([.,]0+)?$'),
    'cod_perfil_periodo', 'string', COUNT_IF(`cod_perfil_periodo` IS NULL), COUNT_IF(`cod_perfil_periodo` IS NOT NULL AND lower(trim(`cod_perfil_periodo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_perfil_periodo`) RLIKE '^0+([.,]0+)?$'),
    'qt_margem_seguranca', 'decimal(5,0)', COUNT_IF(`qt_margem_seguranca` IS NULL), 0L, COUNT_IF(`qt_margem_seguranca` = 0),
    'tp_suprimento_especial', 'string', COUNT_IF(`tp_suprimento_especial` IS NULL), COUNT_IF(`tp_suprimento_especial` IS NOT NULL AND lower(trim(`tp_suprimento_especial`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_suprimento_especial`) RLIKE '^0+([.,]0+)?$'),
    'ind_utilizacao_quotizacao', 'string', COUNT_IF(`ind_utilizacao_quotizacao` IS NULL), COUNT_IF(`ind_utilizacao_quotizacao` IS NOT NULL AND lower(trim(`ind_utilizacao_quotizacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_utilizacao_quotizacao`) RLIKE '^0+([.,]0+)?$'),
    'ind_aap', 'string', COUNT_IF(`ind_aap` IS NULL), COUNT_IF(`ind_aap` IS NOT NULL AND lower(trim(`ind_aap`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_aap`) RLIKE '^0+([.,]0+)?$'),
    'qt_tempo_producao_interna', 'decimal(5,0)', COUNT_IF(`qt_tempo_producao_interna` IS NULL), 0L, COUNT_IF(`qt_tempo_producao_interna` = 0),
    'qt_estoque_seguranca', 'decimal(18,3)', COUNT_IF(`qt_estoque_seguranca` IS NULL), 0L, COUNT_IF(`qt_estoque_seguranca` = 0),
    'qt_estoque_minimo_seguranca', 'decimal(18,3)', COUNT_IF(`qt_estoque_minimo_seguranca` IS NULL), 0L, COUNT_IF(`qt_estoque_minimo_seguranca` = 0),
    'pct_grau_atendimento', 'decimal(5,2)', COUNT_IF(`pct_grau_atendimento` IS NULL), 0L, COUNT_IF(`pct_grau_atendimento` = 0),
    'qt_dias_prazo_entrega_previsto', 'decimal(5,0)', COUNT_IF(`qt_dias_prazo_entrega_previsto` IS NULL), 0L, COUNT_IF(`qt_dias_prazo_entrega_previsto` = 0),
    'qt_dias_processamento_entrada', 'decimal(5,0)', COUNT_IF(`qt_dias_processamento_entrada` IS NULL), 0L, COUNT_IF(`qt_dias_processamento_entrada` = 0),
    'ind_marcado_exclusao', 'string', COUNT_IF(`ind_marcado_exclusao` IS NULL), COUNT_IF(`ind_marcado_exclusao` IS NOT NULL AND lower(trim(`ind_marcado_exclusao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_marcado_exclusao`) RLIKE '^0+([.,]0+)?$')
  ) AS (coluna, tipo, nulos, vazios, zeros) FROM prd_procurement.corp_curated.vw_zglmm159_m2)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros,
       t.total - p.nulos - p.vazios - p.zeros AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros <= 0 THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END AS veredito,
       CASE WHEN p.coluna IN ('ind_remessa_jit', 'ind_entrada_lotes', 'cod_calendario_planejamento', 'cod_perfil_cobertura', 'cod_margem_seguranca', 'cod_perfil_periodo', 'ind_utilizacao_quotizacao', 'ind_aap', 'qt_estoque_minimo_seguranca', 'pct_grau_atendimento') THEN 'declarada sem fonte' ELSE '' END AS observacao
FROM perf p CROSS JOIN t ORDER BY veredito, pct_util, coluna;

## 6.1 Colunas declaradas sem fonte

Confirma que as 10 colunas marcadas *"Sem fonte nas cinco cleansed"* estão realmente nulas.
**Na comparação com o SAP, elas devem ser excluídas** e registradas como limitação de escopo.

In [0]:
-- 6.1 COLUNAS SEM FONTE
SELECT 'ind_remessa_jit' AS coluna, COUNT_IF(`ind_remessa_jit` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`ind_remessa_jit` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m2
UNION ALL
SELECT 'ind_entrada_lotes' AS coluna, COUNT_IF(`ind_entrada_lotes` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`ind_entrada_lotes` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m2
UNION ALL
SELECT 'cod_calendario_planejamento' AS coluna, COUNT_IF(`cod_calendario_planejamento` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`cod_calendario_planejamento` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m2
UNION ALL
SELECT 'cod_perfil_cobertura' AS coluna, COUNT_IF(`cod_perfil_cobertura` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`cod_perfil_cobertura` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m2
UNION ALL
SELECT 'cod_margem_seguranca' AS coluna, COUNT_IF(`cod_margem_seguranca` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`cod_margem_seguranca` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m2
UNION ALL
SELECT 'cod_perfil_periodo' AS coluna, COUNT_IF(`cod_perfil_periodo` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`cod_perfil_periodo` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m2
UNION ALL
SELECT 'ind_utilizacao_quotizacao' AS coluna, COUNT_IF(`ind_utilizacao_quotizacao` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`ind_utilizacao_quotizacao` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m2
UNION ALL
SELECT 'ind_aap' AS coluna, COUNT_IF(`ind_aap` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`ind_aap` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m2
UNION ALL
SELECT 'qt_estoque_minimo_seguranca' AS coluna, COUNT_IF(`qt_estoque_minimo_seguranca` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`qt_estoque_minimo_seguranca` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m2
UNION ALL
SELECT 'pct_grau_atendimento' AS coluna, COUNT_IF(`pct_grau_atendimento` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`pct_grau_atendimento` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m2;

## 7. Cardinalidade

In [0]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_zglmm159_m2),
card AS (SELECT stack(28,
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_empresa', 'string', approx_count_distinct(`cod_empresa`),
    'tp_suprimento', 'string', approx_count_distinct(`tp_suprimento`),
    'tp_grupo_determinacao_estoque', 'string', approx_count_distinct(`tp_grupo_determinacao_estoque`),
    'ind_remessa_jit', 'string', approx_count_distinct(`ind_remessa_jit`),
    'cod_chave_horizonte', 'string', approx_count_distinct(`cod_chave_horizonte`),
    'ind_entrada_lotes', 'string', approx_count_distinct(`ind_entrada_lotes`),
    'ind_coproduto', 'string', approx_count_distinct(`ind_coproduto`),
    'cod_deposito_suprimento_externo', 'string', approx_count_distinct(`cod_deposito_suprimento_externo`),
    'cod_deposito_producao', 'string', approx_count_distinct(`cod_deposito_producao`),
    'cod_calendario_planejamento', 'string', approx_count_distinct(`cod_calendario_planejamento`),
    'ind_baixa_explosao', 'string', approx_count_distinct(`ind_baixa_explosao`),
    'cod_perfil_cobertura', 'string', approx_count_distinct(`cod_perfil_cobertura`),
    'ind_material_granel', 'string', approx_count_distinct(`ind_material_granel`),
    'cod_margem_seguranca', 'string', approx_count_distinct(`cod_margem_seguranca`),
    'cod_perfil_periodo', 'string', approx_count_distinct(`cod_perfil_periodo`),
    'qt_margem_seguranca', 'decimal(5,0)', approx_count_distinct(`qt_margem_seguranca`),
    'tp_suprimento_especial', 'string', approx_count_distinct(`tp_suprimento_especial`),
    'ind_utilizacao_quotizacao', 'string', approx_count_distinct(`ind_utilizacao_quotizacao`),
    'ind_aap', 'string', approx_count_distinct(`ind_aap`),
    'qt_tempo_producao_interna', 'decimal(5,0)', approx_count_distinct(`qt_tempo_producao_interna`),
    'qt_estoque_seguranca', 'decimal(18,3)', approx_count_distinct(`qt_estoque_seguranca`),
    'qt_estoque_minimo_seguranca', 'decimal(18,3)', approx_count_distinct(`qt_estoque_minimo_seguranca`),
    'pct_grau_atendimento', 'decimal(5,2)', approx_count_distinct(`pct_grau_atendimento`),
    'qt_dias_prazo_entrega_previsto', 'decimal(5,0)', approx_count_distinct(`qt_dias_prazo_entrega_previsto`),
    'qt_dias_processamento_entrada', 'decimal(5,0)', approx_count_distinct(`qt_dias_processamento_entrada`),
    'ind_marcado_exclusao', 'string', approx_count_distinct(`ind_marcado_exclusao`)
  ) AS (coluna, tipo, distintos) FROM prd_procurement.corp_curated.vw_zglmm159_m2)
SELECT c.coluna, c.tipo, c.distintos, ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1 THEN '1. CONSTANTE' WHEN c.distintos <= 3 THEN '2. muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador' ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t ORDER BY c.distintos;

## 8. Domínio das categóricas

Top 8 valores de cada uma.

In [0]:
-- 8. DOMINIO
(SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY `cod_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_suprimento' AS coluna, CAST(`tp_suprimento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY `tp_suprimento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_suprimento_especial' AS coluna, CAST(`tp_suprimento_especial` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY `tp_suprimento_especial` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_grupo_determinacao_estoque' AS coluna, CAST(`tp_grupo_determinacao_estoque` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY `tp_grupo_determinacao_estoque` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_chave_horizonte' AS coluna, CAST(`cod_chave_horizonte` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY `cod_chave_horizonte` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_coproduto' AS coluna, CAST(`ind_coproduto` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY `ind_coproduto` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_baixa_explosao' AS coluna, CAST(`ind_baixa_explosao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY `ind_baixa_explosao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_material_granel' AS coluna, CAST(`ind_material_granel` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY `ind_material_granel` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_deposito_suprimento_externo' AS coluna, CAST(`cod_deposito_suprimento_externo` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY `cod_deposito_suprimento_externo` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_deposito_producao' AS coluna, CAST(`cod_deposito_producao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY `cod_deposito_producao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_marcado_exclusao' AS coluna, CAST(`ind_marcado_exclusao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY `ind_marcado_exclusao` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 10. Perfil numérico

Todos `decimal` — conciliação exata, sem tolerância.

In [0]:
-- 10. PERFIL NUMERICO
SELECT * FROM (SELECT stack(7,
    'qt_margem_seguranca', 'decimal(5,0)', COUNT(`qt_margem_seguranca`), CAST(MIN(`qt_margem_seguranca`) AS DOUBLE), CAST(MAX(`qt_margem_seguranca`) AS DOUBLE), CAST(AVG(`qt_margem_seguranca`) AS DOUBLE), CAST(percentile_approx(`qt_margem_seguranca`, 0.5) AS DOUBLE), COUNT_IF(`qt_margem_seguranca` < 0), COUNT_IF(`qt_margem_seguranca` = 0),
    'qt_tempo_producao_interna', 'decimal(5,0)', COUNT(`qt_tempo_producao_interna`), CAST(MIN(`qt_tempo_producao_interna`) AS DOUBLE), CAST(MAX(`qt_tempo_producao_interna`) AS DOUBLE), CAST(AVG(`qt_tempo_producao_interna`) AS DOUBLE), CAST(percentile_approx(`qt_tempo_producao_interna`, 0.5) AS DOUBLE), COUNT_IF(`qt_tempo_producao_interna` < 0), COUNT_IF(`qt_tempo_producao_interna` = 0),
    'qt_estoque_seguranca', 'decimal(18,3)', COUNT(`qt_estoque_seguranca`), CAST(MIN(`qt_estoque_seguranca`) AS DOUBLE), CAST(MAX(`qt_estoque_seguranca`) AS DOUBLE), CAST(AVG(`qt_estoque_seguranca`) AS DOUBLE), CAST(percentile_approx(`qt_estoque_seguranca`, 0.5) AS DOUBLE), COUNT_IF(`qt_estoque_seguranca` < 0), COUNT_IF(`qt_estoque_seguranca` = 0),
    'qt_estoque_minimo_seguranca', 'decimal(18,3)', COUNT(`qt_estoque_minimo_seguranca`), CAST(MIN(`qt_estoque_minimo_seguranca`) AS DOUBLE), CAST(MAX(`qt_estoque_minimo_seguranca`) AS DOUBLE), CAST(AVG(`qt_estoque_minimo_seguranca`) AS DOUBLE), CAST(percentile_approx(`qt_estoque_minimo_seguranca`, 0.5) AS DOUBLE), COUNT_IF(`qt_estoque_minimo_seguranca` < 0), COUNT_IF(`qt_estoque_minimo_seguranca` = 0),
    'pct_grau_atendimento', 'decimal(5,2)', COUNT(`pct_grau_atendimento`), CAST(MIN(`pct_grau_atendimento`) AS DOUBLE), CAST(MAX(`pct_grau_atendimento`) AS DOUBLE), CAST(AVG(`pct_grau_atendimento`) AS DOUBLE), CAST(percentile_approx(`pct_grau_atendimento`, 0.5) AS DOUBLE), COUNT_IF(`pct_grau_atendimento` < 0), COUNT_IF(`pct_grau_atendimento` = 0),
    'qt_dias_prazo_entrega_previsto', 'decimal(5,0)', COUNT(`qt_dias_prazo_entrega_previsto`), CAST(MIN(`qt_dias_prazo_entrega_previsto`) AS DOUBLE), CAST(MAX(`qt_dias_prazo_entrega_previsto`) AS DOUBLE), CAST(AVG(`qt_dias_prazo_entrega_previsto`) AS DOUBLE), CAST(percentile_approx(`qt_dias_prazo_entrega_previsto`, 0.5) AS DOUBLE), COUNT_IF(`qt_dias_prazo_entrega_previsto` < 0), COUNT_IF(`qt_dias_prazo_entrega_previsto` = 0),
    'qt_dias_processamento_entrada', 'decimal(5,0)', COUNT(`qt_dias_processamento_entrada`), CAST(MIN(`qt_dias_processamento_entrada`) AS DOUBLE), CAST(MAX(`qt_dias_processamento_entrada`) AS DOUBLE), CAST(AVG(`qt_dias_processamento_entrada`) AS DOUBLE), CAST(percentile_approx(`qt_dias_processamento_entrada`, 0.5) AS DOUBLE), COUNT_IF(`qt_dias_processamento_entrada` < 0), COUNT_IF(`qt_dias_processamento_entrada` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros) FROM prd_procurement.corp_curated.vw_zglmm159_m2)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

In [0]:
-- 11. TOTAIS
SELECT * FROM (SELECT stack(7,
    'qt_margem_seguranca', CAST(ROUND(SUM(`qt_margem_seguranca`), 3) AS STRING), COUNT(`qt_margem_seguranca`),
    'qt_tempo_producao_interna', CAST(ROUND(SUM(`qt_tempo_producao_interna`), 3) AS STRING), COUNT(`qt_tempo_producao_interna`),
    'qt_estoque_seguranca', CAST(ROUND(SUM(`qt_estoque_seguranca`), 3) AS STRING), COUNT(`qt_estoque_seguranca`),
    'qt_estoque_minimo_seguranca', CAST(ROUND(SUM(`qt_estoque_minimo_seguranca`), 3) AS STRING), COUNT(`qt_estoque_minimo_seguranca`),
    'pct_grau_atendimento', CAST(ROUND(SUM(`pct_grau_atendimento`), 3) AS STRING), COUNT(`pct_grau_atendimento`),
    'qt_dias_prazo_entrega_previsto', CAST(ROUND(SUM(`qt_dias_prazo_entrega_previsto`), 3) AS STRING), COUNT(`qt_dias_prazo_entrega_previsto`),
    'qt_dias_processamento_entrada', CAST(ROUND(SUM(`qt_dias_processamento_entrada`), 3) AS STRING), COUNT(`qt_dias_processamento_entrada`)
  ) AS (coluna, total, linhas_preenchidas) FROM prd_procurement.corp_curated.vw_zglmm159_m2)
ORDER BY coluna;

## 12. Datas

In [0]:
-- 12. DATAS
SELECT 'Esta view nao possui colunas de data' AS observacao;

## 13. Códigos e zeros à esquerda

Na validação anterior, `cod_material` vinha com zeros à esquerda e o extrato SAP sem — normalizar os dois lados.

In [0]:
-- 13. CODIGOS
SELECT coluna, vazios, len_min, len_max, com_zeros_esq, distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes
FROM (SELECT stack(5,
    'cod_material', COUNT_IF(`cod_material` IS NULL OR trim(`cod_material`) = ''), MIN(length(trim(`cod_material`))), MAX(length(trim(`cod_material`))), COUNT_IF(trim(`cod_material`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_material`)), COUNT(DISTINCT regexp_replace(trim(`cod_material`), '^0+', '')),
    'cod_centro', COUNT_IF(`cod_centro` IS NULL OR trim(`cod_centro`) = ''), MIN(length(trim(`cod_centro`))), MAX(length(trim(`cod_centro`))), COUNT_IF(trim(`cod_centro`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_centro`)), COUNT(DISTINCT regexp_replace(trim(`cod_centro`), '^0+', '')),
    'cod_empresa', COUNT_IF(`cod_empresa` IS NULL OR trim(`cod_empresa`) = ''), MIN(length(trim(`cod_empresa`))), MAX(length(trim(`cod_empresa`))), COUNT_IF(trim(`cod_empresa`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_empresa`)), COUNT(DISTINCT regexp_replace(trim(`cod_empresa`), '^0+', '')),
    'cod_deposito_suprimento_externo', COUNT_IF(`cod_deposito_suprimento_externo` IS NULL OR trim(`cod_deposito_suprimento_externo`) = ''), MIN(length(trim(`cod_deposito_suprimento_externo`))), MAX(length(trim(`cod_deposito_suprimento_externo`))), COUNT_IF(trim(`cod_deposito_suprimento_externo`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_deposito_suprimento_externo`)), COUNT(DISTINCT regexp_replace(trim(`cod_deposito_suprimento_externo`), '^0+', '')),
    'cod_deposito_producao', COUNT_IF(`cod_deposito_producao` IS NULL OR trim(`cod_deposito_producao`) = ''), MIN(length(trim(`cod_deposito_producao`))), MAX(length(trim(`cod_deposito_producao`))), COUNT_IF(trim(`cod_deposito_producao`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_deposito_producao`)), COUNT(DISTINCT regexp_replace(trim(`cod_deposito_producao`), '^0+', ''))
  ) AS (coluna, vazios, len_min, len_max, com_zeros_esq, distintos_bruto, distintos_sem_zeros) FROM prd_procurement.corp_curated.vw_zglmm159_m2)
ORDER BY coluna;

## 14. Checksum de linha

Se `linhas` > `linhas_unicas`, há registros 100% idênticos.

In [0]:
-- 14. CHECKSUM
SELECT COUNT(*) AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`tp_suprimento` AS STRING), ''), COALESCE(CAST(`tp_grupo_determinacao_estoque` AS STRING), ''), COALESCE(CAST(`ind_remessa_jit` AS STRING), ''), COALESCE(CAST(`cod_chave_horizonte` AS STRING), ''), COALESCE(CAST(`ind_entrada_lotes` AS STRING), ''), COALESCE(CAST(`ind_coproduto` AS STRING), ''), COALESCE(CAST(`cod_deposito_suprimento_externo` AS STRING), ''), COALESCE(CAST(`cod_deposito_producao` AS STRING), ''), COALESCE(CAST(`cod_calendario_planejamento` AS STRING), ''), COALESCE(CAST(`ind_baixa_explosao` AS STRING), ''), COALESCE(CAST(`cod_perfil_cobertura` AS STRING), ''), COALESCE(CAST(`ind_material_granel` AS STRING), ''), COALESCE(CAST(`cod_margem_seguranca` AS STRING), ''), COALESCE(CAST(`cod_perfil_periodo` AS STRING), ''), COALESCE(CAST(`qt_margem_seguranca` AS STRING), ''), COALESCE(CAST(`tp_suprimento_especial` AS STRING), ''), COALESCE(CAST(`ind_utilizacao_quotizacao` AS STRING), ''), COALESCE(CAST(`ind_aap` AS STRING), ''), COALESCE(CAST(`qt_tempo_producao_interna` AS STRING), ''), COALESCE(CAST(`qt_estoque_seguranca` AS STRING), ''), COALESCE(CAST(`qt_estoque_minimo_seguranca` AS STRING), ''), COALESCE(CAST(`pct_grau_atendimento` AS STRING), ''), COALESCE(CAST(`qt_dias_prazo_entrega_previsto` AS STRING), ''), COALESCE(CAST(`qt_dias_processamento_entrada` AS STRING), ''), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '')))) AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`tp_suprimento` AS STRING), ''), COALESCE(CAST(`tp_grupo_determinacao_estoque` AS STRING), ''), COALESCE(CAST(`ind_remessa_jit` AS STRING), ''), COALESCE(CAST(`cod_chave_horizonte` AS STRING), ''), COALESCE(CAST(`ind_entrada_lotes` AS STRING), ''), COALESCE(CAST(`ind_coproduto` AS STRING), ''), COALESCE(CAST(`cod_deposito_suprimento_externo` AS STRING), ''), COALESCE(CAST(`cod_deposito_producao` AS STRING), ''), COALESCE(CAST(`cod_calendario_planejamento` AS STRING), ''), COALESCE(CAST(`ind_baixa_explosao` AS STRING), ''), COALESCE(CAST(`cod_perfil_cobertura` AS STRING), ''), COALESCE(CAST(`ind_material_granel` AS STRING), ''), COALESCE(CAST(`cod_margem_seguranca` AS STRING), ''), COALESCE(CAST(`cod_perfil_periodo` AS STRING), ''), COALESCE(CAST(`qt_margem_seguranca` AS STRING), ''), COALESCE(CAST(`tp_suprimento_especial` AS STRING), ''), COALESCE(CAST(`ind_utilizacao_quotizacao` AS STRING), ''), COALESCE(CAST(`ind_aap` AS STRING), ''), COALESCE(CAST(`qt_tempo_producao_interna` AS STRING), ''), COALESCE(CAST(`qt_estoque_seguranca` AS STRING), ''), COALESCE(CAST(`qt_estoque_minimo_seguranca` AS STRING), ''), COALESCE(CAST(`pct_grau_atendimento` AS STRING), ''), COALESCE(CAST(`qt_dias_prazo_entrega_previsto` AS STRING), ''), COALESCE(CAST(`qt_dias_processamento_entrada` AS STRING), ''), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '')))) AS linhas_100pct_identicas
FROM prd_procurement.corp_curated.vw_zglmm159_m2;

## 15. Amostra

In [0]:
-- 15. AMOSTRA
SELECT * FROM prd_procurement.corp_curated.vw_zglmm159_m2
ORDER BY `cod_material`, `cod_centro`
LIMIT 20;

## 16. Distribuição interna

Inclui `ind_marcado_exclusao` — mostra quanto da view são itens marcados para eliminação.

In [0]:
-- 16. DISTRIBUICAO POR cod_empresa
SELECT COALESCE(NULLIF(trim(`cod_empresa`), ''), '(vazio)') AS `cod_empresa`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_m2), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY 1 ORDER BY linhas DESC LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR tp_suprimento
SELECT COALESCE(NULLIF(trim(`tp_suprimento`), ''), '(vazio)') AS `tp_suprimento`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_m2), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY 1 ORDER BY linhas DESC LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR tp_suprimento_especial
SELECT COALESCE(NULLIF(trim(`tp_suprimento_especial`), ''), '(vazio)') AS `tp_suprimento_especial`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_m2), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY 1 ORDER BY linhas DESC LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR ind_marcado_exclusao
SELECT COALESCE(NULLIF(trim(`ind_marcado_exclusao`), ''), '(vazio)') AS `ind_marcado_exclusao`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_m2), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY 1 ORDER BY linhas DESC LIMIT 40;

## 18. Análises específicas — M2 (MRP 2)

### 18.1 Tipo de suprimento

Domínio SAP (BESKZ): **E** = produção interna, **F** = aquisição externa, **X** = ambos.

In [0]:
-- 18.1 TIPO DE SUPRIMENTO x SUPRIMENTO ESPECIAL
SELECT COALESCE(NULLIF(trim(tp_suprimento),''),'(vazio)') AS tp_suprimento,
       COALESCE(NULLIF(trim(tp_suprimento_especial),''),'(vazio)') AS tp_suprimento_especial,
       COUNT(*) AS linhas,
       CASE WHEN trim(COALESCE(tp_suprimento,'')) IN ('E','F','X','') THEN 'dominio esperado' ELSE 'FORA DO DOMINIO' END AS veredito
FROM prd_procurement.corp_curated.vw_zglmm159_m2 GROUP BY 1, 2, trim(COALESCE(tp_suprimento,'')) ORDER BY linhas DESC LIMIT 30;

### 18.2 Prazos e estoque de segurança

In [0]:
-- 18.2 PRAZOS E ESTOQUE DE SEGURANCA
SELECT 'prazo de entrega previsto negativo' AS regra, COUNT_IF(qt_dias_prazo_entrega_previsto < 0) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m2
UNION ALL SELECT 'processamento de entrada negativo', COUNT_IF(qt_dias_processamento_entrada < 0) FROM prd_procurement.corp_curated.vw_zglmm159_m2
UNION ALL SELECT 'estoque de seguranca negativo', COUNT_IF(qt_estoque_seguranca < 0) FROM prd_procurement.corp_curated.vw_zglmm159_m2
UNION ALL SELECT 'aquisicao externa (F) sem prazo de entrega', COUNT_IF(trim(tp_suprimento) = 'F' AND COALESCE(qt_dias_prazo_entrega_previsto,0) = 0) FROM prd_procurement.corp_curated.vw_zglmm159_m2
UNION ALL SELECT 'producao interna (E) sem tempo de producao', COUNT_IF(trim(tp_suprimento) = 'E' AND COALESCE(qt_tempo_producao_interna,0) = 0) FROM prd_procurement.corp_curated.vw_zglmm159_m2
UNION ALL SELECT 'margem de seguranca preenchida sem chave de horizonte', COUNT_IF(COALESCE(qt_margem_seguranca,0) > 0 AND trim(COALESCE(cod_chave_horizonte,'')) = '') FROM prd_procurement.corp_curated.vw_zglmm159_m2
ORDER BY linhas DESC;

### 18.9 🔗 Cruzamento entre as 3 views

A ZGLMM159 foi dividida em C1, M1 e M2. As três deveriam cobrir o **mesmo universo de
`material + centro`** e concordar na empresa. Esta célula mede a cobertura cruzada — divergência
aqui indica material que vai "sumir" quando as três forem reunidas no relatório final.

> Na validação anterior, **434 materiais existiam no SAP e não no Datalake**. Esta análise
> ajuda a localizar em qual das três views eles se perdem.

In [0]:
-- 18.9 COBERTURA DE MATERIAL + CENTRO ENTRE C1, M1, M2
WITH c1 AS (SELECT DISTINCT cod_material, cod_centro FROM prd_procurement.corp_curated.vw_zglmm159_c1),
     m1 AS (SELECT DISTINCT cod_material, cod_centro FROM prd_procurement.corp_curated.vw_zglmm159_m1),
     m2 AS (SELECT DISTINCT cod_material, cod_centro FROM prd_procurement.corp_curated.vw_zglmm159_m2),
     u  AS (SELECT * FROM c1 UNION SELECT * FROM m1 UNION SELECT * FROM m2),
     x  AS (SELECT u.cod_material, u.cod_centro,
                   c1.cod_material IS NOT NULL AS em_c1,
                   m1.cod_material IS NOT NULL AS em_m1,
                   m2.cod_material IS NOT NULL AS em_m2
            FROM u
            LEFT JOIN c1 ON u.cod_material = c1.cod_material AND u.cod_centro = c1.cod_centro
            LEFT JOIN m1 ON u.cod_material = m1.cod_material AND u.cod_centro = m1.cod_centro
            LEFT JOIN m2 ON u.cod_material = m2.cod_material AND u.cod_centro = m2.cod_centro)
SELECT CONCAT(CASE WHEN em_c1 THEN 'C1 ' ELSE '-- ' END,
              CASE WHEN em_m1 THEN 'M1 ' ELSE '-- ' END,
              CASE WHEN em_m2 THEN 'M2'  ELSE '--' END) AS presenca,
       COUNT(*) AS material_centro,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM x GROUP BY 1 ORDER BY material_centro DESC;

In [0]:
-- 18.9b EMPRESA E INDICADOR DE EXCLUSAO CONSISTENTES ENTRE AS VIEWS
WITH a AS (SELECT DISTINCT cod_material, cod_centro, cod_empresa, ind_marcado_exclusao FROM prd_procurement.corp_curated.vw_zglmm159_m2),
     b AS (SELECT DISTINCT cod_material, cod_centro, cod_empresa, ind_marcado_exclusao FROM prd_procurement.corp_curated.vw_zglmm159_c1),
     c AS (SELECT DISTINCT cod_material, cod_centro, cod_empresa, ind_marcado_exclusao FROM prd_procurement.corp_curated.vw_zglmm159_m1)
SELECT 'M2 x C1' AS par,
       COUNT(*) AS material_centro_em_comum,
       COUNT_IF(COALESCE(a.cod_empresa,'') <> COALESCE(b.cod_empresa,'')) AS empresa_diverge,
       COUNT_IF(COALESCE(a.ind_marcado_exclusao,'') <> COALESCE(b.ind_marcado_exclusao,'')) AS exclusao_diverge
  FROM a JOIN b ON a.cod_material = b.cod_material AND a.cod_centro = b.cod_centro
UNION ALL
SELECT 'M2 x M1',
       COUNT(*),
       COUNT_IF(COALESCE(a.cod_empresa,'') <> COALESCE(c.cod_empresa,'')),
       COUNT_IF(COALESCE(a.ind_marcado_exclusao,'') <> COALESCE(c.ind_marcado_exclusao,''))
  FROM a JOIN c ON a.cod_material = c.cod_material AND a.cod_centro = c.cod_centro;

## 19. Amostra ampliada

A extração completa fica nos notebooks de cenário.

In [0]:
-- 19. AMOSTRA AMPLIADA
SELECT * FROM prd_procurement.corp_curated.vw_zglmm159_m2
ORDER BY `cod_material`, `cod_centro`
LIMIT 500;

## 20. Resumo

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_zglmm159_m2),
g AS (
  SELECT 'cod_material' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material` FROM prd_procurement.corp_curated.vw_zglmm159_m2)
UNION ALL
  SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_centro` FROM prd_procurement.corp_curated.vw_zglmm159_m2)
UNION ALL
  SELECT 'cod_material + cod_centro + cod_empresa' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_empresa` FROM prd_procurement.corp_curated.vw_zglmm159_m2)
)
SELECT 'IDENTIFICACAO' AS bloco, 'transacao' AS item, 'ZGLMM159 - M2' AS valor
UNION ALL SELECT 'IDENTIFICACAO', 'ambiente', 'PRD'
UNION ALL SELECT 'IDENTIFICACAO', 'tabela', 'vw_zglmm159_m2'
UNION ALL SELECT 'IDENTIFICACAO', 'linhas', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'IDENTIFICACAO', 'colunas', '28 (10 declaradas sem fonte)'
UNION ALL SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ', CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total), 'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Rodar os **3 notebooks** (C1, M1, M2) e comparar a seção 18.9 entre eles.
2. Escolher o recorte de teste pela **seção 3/3.1**.
3. Gerar os notebooks de cenário e extrair a ZGLMM159 no **SAP PRD** com o mesmo recorte e na mesma data.

### Checklist antes de comparar

- [ ] Chave real confirmada (seção 4)
- [ ] Colunas sem fonte excluídas da comparação (seção 6.1)
- [ ] Zeros à esquerda normalizados (seção 13)
- [ ] Itens marcados para exclusão tratados conforme o filtro do SAP (seção 16)
- [ ] Cobertura C1 × M1 × M2 conferida (seção 18.9)
